In [12]:
from pathlib import Path

sample_dir = Path("/data/bvcoe2026/CMS-OD/samples")

print("Directory exists:", sample_dir.exists())

if sample_dir.exists():
    root_files = sorted(sample_dir.glob("*.root"))
    print("Number of ROOT files:", len(root_files))

    for file in root_files:
        print(file.name)
else:
    print("Sample directory not found.")

Directory exists: False
Sample directory not found.


In [ ]:
import uproot

In [ ]:
import uproot

file_path = str(root_files[0])

with uproot.open(file_path) as f:
    print("Objects inside file:")
    print(f.keys())

In [ ]:
with uproot.open(file_path) as f:
    tree = f["Events"]

    print("Total events:", tree.num_entries)
    print("Available branches:")
    print(tree.keys())

In [ ]:
with uproot.open(file_path) as f:
    tree = f["Events"]

    for name in tree.keys():
        if any(x in name for x in ["Muon_pt", "Electron_pt", "Jet_pt", "MET_pt"]):
            print(name)

In [ ]:
import uproot
import matplotlib.pyplot as plt

with uproot.open(file_path) as f:
    tree = f["Events"]
    values = tree["Lepton1_pt"].array(
        entry_stop=10000,
        library="ak"
    )

plt.figure(figsize=(8, 5))
plt.hist(values, bins=50, histtype="step")
plt.xlabel("Leading lepton transverse momentum (GeV)")
plt.ylabel("Number of leptons")
plt.title("Leading Lepton pT Distribution")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import uproot
import awkward as ak
import matplotlib.pyplot as plt

variables = ["Lepton1_pt", "Lepton2_pt", "Jet1_pt", "Jet2_pt", "PuppiMET_pt"]

with uproot.open(file_path) as f:
    tree = f["Events"]

    for variable in variables:
        if variable not in tree.keys():
            print("Skipping missing branch:", variable)
            continue

        arr = tree[variable].array(
            entry_stop=10000,
            library="ak"
        )
        values = ak.to_numpy(arr)

        plt.figure(figsize=(8, 5))
        plt.hist(values, bins=50, histtype="step")
        plt.xlabel(f"{variable} (GeV)")
        plt.ylabel("Number of events")
        plt.title(f"{variable} Distribution")
        plt.grid(alpha=0.3)
        plt.show()

In [13]:
from pathlib import Path
import math
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sample_dir = Path("/data/bvcoe2026/CMS-OD/samples")

sample_files = {
    "DY": sample_dir / "DY-train.root",
    "Higgs": sample_dir / "Higgs-train.root",
    "Top": sample_dir / "Top-train.root",
    "WW": sample_dir / "WW-train.root",
}

# Branch names that exist in the flat ntuple (missing ones are skipped automatically)
mvaVariables = [
    "Lepton1_pt", "Lepton1_eta", "Lepton1_phi",
    "Lepton2_pt", "Lepton2_eta", "Lepton2_phi",
    "Jet1_pt", "Jet1_eta", "Jet1_phi",
    "Jet2_pt", "Jet2_eta", "Jet2_phi",
    "PuppiMET_pt", "PuppiMET_phi",
    "njet", "jet_cat",
    "mll", "ptll", "mth", "mtw2", "mjj", "dphill",
]

output_dir = Path("superimposed_plots")
output_dir.mkdir(parents=True, exist_ok=True)

N_COLS = 6          # panels per row
N_BINS = 40
ENTRY_STOP = 10000  # events read per sample
PLACEHOLDER_CUT = -999  # drop sentinel values such as -9999

colors = {"DY": "tab:green", "Higgs": "tab:red",
          "Top": "goldenrod", "WW": "tab:blue"}

# ---- Locate sample files ----
available_samples = {}
for label, path in sample_files.items():
    if path.exists():
        available_samples[label] = path
        print(f"Found {label}: {path}")
    else:
        print(f"MISSING {label}: {path}")

if len(available_samples) < 2:
    raise FileNotFoundError("Need at least two sample files.")

# ---- Open trees ----
trees = {}
for label, path in available_samples.items():
    root_file = uproot.open(path)
    if "Events" not in root_file:
        print(f"Skipping {label}: no Events tree")
        root_file.close()
        continue
    trees[label] = (root_file, root_file["Events"])

# ---- Keep only variables present in at least two samples ----
keys = {label: set(tree.keys()) for label, (_, tree) in trees.items()}
plot_vars = []
for v in mvaVariables:
    n_found = sum(v in k for k in keys.values())
    if n_found >= 2:
        plot_vars.append(v)
    else:
        print(f"Skipping {v}: found in {n_found} sample(s)")

if not plot_vars:
    raise ValueError("None of the requested variables were found.")

# ---- Grid layout (+1 slot for the legend) ----
n_slots = len(plot_vars) + 1
n_rows = math.ceil(n_slots / N_COLS)

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(4 * N_COLS, 3 * n_rows))
axes = axes.ravel()

for panel, variable in enumerate(plot_vars):
    # Read each sample's values for this variable
    data = {}
    for label, (root_file, tree) in trees.items():
        if variable not in keys[label]:
            continue
        try:
            arr = tree[variable].array(entry_stop=ENTRY_STOP, library="ak")
            vals = ak.to_numpy(ak.flatten(arr, axis=None)).astype(float)
            vals = vals[np.isfinite(vals) & (vals > PLACEHOLDER_CUT)]
            if len(vals) > 0:
                data[label] = vals
        except Exception as exc:
            print(f"Could not read {variable} from {label}: {exc}")

    ax = axes[panel]
    if len(data) < 2:
        ax.axis("off")
        continue

    # Common bin edges for all samples (clip extreme outliers)
    allvals = np.concatenate(list(data.values()))
    lo, hi = np.percentile(allvals, [0.5, 99.5])
    if lo == hi:
        lo, hi = lo - 0.5, hi + 0.5
    bins = np.linspace(lo, hi, N_BINS + 1)

    for label, vals in data.items():
        ax.hist(vals, bins=bins, histtype="step", density=True,
                linewidth=1.3, color=colors.get(label))

    ax.set_xlabel(variable, fontsize=9)
    ax.set_title(variable, fontsize=10)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.25)
    if panel % N_COLS == 0:
        ax.set_ylabel("Normalized density", fontsize=9)

# ---- Legend in the last slot ----
legend_ax = axes[-1]
legend_ax.axis("off")
handles = [Line2D([0], [0], color=colors.get(l), lw=2, label=l)
           for l in trees.keys()]
legend_ax.legend(handles=handles, loc="center", fontsize=12, title="Samples")

# ---- Hide unused panels between the last plot and the legend ----
for ax in axes[len(plot_vars):-1]:
    ax.axis("off")

fig.suptitle("Training Sample Comparison: All Input Variables",
             fontsize=18, y=1.0)
fig.tight_layout()
fig.savefig(output_dir / "all_variables_grid.png", dpi=150, bbox_inches="tight")
fig.savefig(output_dir / "all_variables_grid.pdf", bbox_inches="tight")
plt.show()

for root_file, tree in trees.values():
    root_file.close()

print(f"\nPlots saved in: {output_dir.resolve()}")

MISSING DY: /data/bvcoe2026/CMS-OD/samples/DY-train.root
MISSING Higgs: /data/bvcoe2026/CMS-OD/samples/Higgs-train.root
MISSING Top: /data/bvcoe2026/CMS-OD/samples/Top-train.root
MISSING WW: /data/bvcoe2026/CMS-OD/samples/WW-train.root


FileNotFoundError: Need at least two sample files.

In [ ]:
import uproot
%pip install seaborn
import seaborn as sns
import numpy as np
import pandas as pd
import uproot
import awkward as ak
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, roc_auc_score

In [ ]:
sample_dir  = Path("/data/bvcoe2026/CMS-OD/samples")
N_EVENTS    = 100_000
N_TREES     = 400
MAX_DEPTH   = 10
RANDOM_SEED = 42

FLAT_FEATURES = [
    # Event-level kinematics
    "dphill", "mll", "mjj", "mth", "mtw2", "ptll",
    "njet",
    "PuppiMET_pt", "PuppiMET_phi",
    # Leptons (leading / subleading)
    "Lepton1_pt", "Lepton1_eta", "Lepton1_phi",
    "Lepton2_pt", "Lepton2_eta", "Lepton2_phi",
    # Jets (leading / subleading)
    "Jet1_pt", "Jet1_eta", "Jet1_phi",
    "Jet2_pt",
]

# No jagged branches in these ntuples; kept empty so downstream code still works
JAGGED_FEATURES = {}

ALL_FEATURES = FLAT_FEATURES + list(JAGGED_FEATURES.keys())

TRAIN_SAMPLES = {
    "Higgs": (sample_dir / "Higgs-train.root", 1),
    "DY":    (sample_dir / "DY-train.root",    0),
    "Top":   (sample_dir / "Top-train.root",   0),
    "WW":    (sample_dir / "WW-train.root",    0),
}
TEST_SAMPLES = {
    "Higgs": (sample_dir / "Higgs-test.root", 1),
    "DY":    (sample_dir / "DY-test.root",    0),
    "Top":   (sample_dir / "Top-test.root",   0),
    "WW":    (sample_dir / "WW-test.root",    0),
}

print("✓ Configuration ready.")
print(f"  Features ({len(ALL_FEATURES)} total):", ALL_FEATURES)
print(f"  Events per sample (train): {N_EVENTS:,}")
print(f"  RF: {N_TREES} trees, max_depth={MAX_DEPTH}")


In [ ]:
import numpy as np
import pandas as pd
import uproot

def load_events(sample_dict, n_events=None):
    frames = []

    for name, (path, label) in sample_dict.items():
        if not path.exists():
            print(f"⚠  Missing file: {path}")
            continue

        print(f"Loading {name:6s} ...", end=" ")

        with uproot.open(path) as f:
            tree = f["Events"]

            # Fail early with a clear message if a configured branch doesn't exist
            missing = [feat for feat in ALL_FEATURES if feat not in tree.keys()]
            if missing:
                raise KeyError(
                    f"{name}: branches not found in {path.name}: {missing}\n"
                    f"Remove them from FLAT_FEATURES or check the spelling."
                )

            df = tree.arrays(
                expressions=ALL_FEATURES,
                entry_stop=n_events,
                library="pd"
            )

        df["label"]  = label
        df["sample"] = name
        df[ALL_FEATURES] = df[ALL_FEATURES].fillna(-999)

        frames.append(df)
        print(f"{len(df):>9,} events  (label={label})")

    return pd.concat(frames, ignore_index=True)

print("✓ load_events() defined.")

In [ ]:
print("Loading TRAINING data")
print("=" * 55)
df_train = load_events(TRAIN_SAMPLES, n_events=N_EVENTS)

print()
print("=" * 55)
print("Loading TEST data")
print("=" * 55)
df_test = load_events(TEST_SAMPLES, n_events=N_EVENTS)

X_train = df_train[ALL_FEATURES].values
y_train = df_train["label"].values

X_test  = df_test[ALL_FEATURES].values
y_test  = df_test["label"].values

print()
print(f"Train: {len(X_train):>9,} events  | "
      f"Signal: {y_train.sum():>7,}  | Background: {(y_train==0).sum():>7,}")
print(f"Test:  {len(X_test):>9,} events  | "
      f"Signal: {y_test.sum():>7,}  | Background: {(y_test==0).sum():>7,}")

print()
print("Events per sample (train):")
print(df_train["sample"].value_counts().to_string())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SENTINEL_CUT = -999   # anything <= this is a placeholder, not a real value

df_feat = pd.DataFrame(X_train, columns=ALL_FEATURES)
df_feat = df_feat.where(df_feat > SENTINEL_CUT)   # sentinels -> NaN
corr    = df_feat.corr(method="pearson", min_periods=100)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.4,
    ax=ax,
    annot_kws={"size": 7},
    cbar_kws={"label": "Pearson r", "shrink": 0.8},
)
ax.set_title("Feature Correlation Matrix (Training Data, placeholders excluded)",
             fontsize=14, pad=14)
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
plt.savefig("feature_correlation_matrix.png", dpi=150)
plt.show()

print("Strongly correlated feature pairs (|r| > 0.7):")
cols = corr.columns
found = False
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if pd.notna(r) and abs(r) > 0.7:
            print(f"  {cols[i]:25s} ↔  {cols[j]:25s}  r = {r:+.3f}")
            found = True
if not found:
    print("  None found — all features have |r| ≤ 0.7")

# Fraction of events with a placeholder, per feature (helps interpret the jet columns)
frac_missing = df_feat.isna().mean().sort_values(ascending=False)
print("\nFraction of placeholder values per feature (top 8):")
print(frac_missing.head(8).round(3).to_string())

In [ ]:
rf = RandomForestClassifier(
    n_estimators     = N_TREES,
    max_depth        = MAX_DEPTH,
    min_samples_leaf = 60,      
    max_features     = "sqrt",  
    class_weight     = "balanced",  
    n_jobs           = -1,     
    random_state     = RANDOM_SEED,
    verbose          = 1,
)

print(f"Training Random Forest: {N_TREES} trees, max_depth={MAX_DEPTH}")
print(f"Training on {len(X_train):,} events with {len(ALL_FEATURES)} features...\n")
rf.fit(X_train, y_train)
score_train = rf.predict_proba(X_train)[:, 1]
score_test  = rf.predict_proba(X_test)[:, 1]

auc_train_quick = roc_auc_score(y_train, score_train)
auc_test_quick  = roc_auc_score(y_test,  score_test)

print(f"\n{'=' * 45}")
print(f"  Train AUC (quick check) : {auc_train_quick:.4f}")
print(f"  Test  AUC               : {auc_test_quick:.4f}")
print(f"  AUC gap                 : {auc_train_quick - auc_test_quick:.4f}")
print(f"{'=' * 45}")
if auc_train_quick - auc_test_quick > 0.05:
    print("  ⚠  Large AUC gap — possible overtraining. Check KS test (Cell G).")
else:
    print("  ✓  AUC gap acceptable — model appears to generalise well.")

In [ ]:
fpr_train, tpr_train, _ = roc_curve(y_train, score_train)
fpr_test,  tpr_test,  _ = roc_curve(y_test,  score_test)

auc_train = auc(fpr_train, tpr_train)
auc_test  = auc(fpr_test,  tpr_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_train, tpr_train, color="royalblue",  lw=2,
        label=f"Train  AUC = {auc_train:.4f}")
ax.plot(fpr_test,  tpr_test,  color="tomato",     lw=2, linestyle="--",
        label=f"Test   AUC = {auc_test:.4f}")
ax.plot([0, 1], [0, 1], "k:", lw=1, label="Random (AUC = 0.5)")
idx = np.argmin(np.abs(fpr_test - 0.30))
ax.scatter(fpr_test[idx], tpr_test[idx], color="tomato", zorder=5,
           label=f"Test @ FPR=0.30 → TPR={tpr_test[idx]:.2f}")

ax.set_xlabel("False Positive Rate (Background efficiency)", fontsize=12)
ax.set_ylabel("True Positive Rate (Signal efficiency)",      fontsize=12)
ax.set_title("ROC Curve — Random Forest: Higgs vs Background", fontsize=13)
ax.legend(fontsize=10, loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()

print(f"Train AUC = {auc_train:.4f}")
print(f"Test  AUC = {auc_test:.4f}")
print(f"AUC gap   = {auc_train - auc_test:.4f}  "
      "(≤0.01 excellent | 0.01–0.05 acceptable | >0.05 overtraining concern)")

In [ ]:
sig_train = score_train[y_train == 1]
bkg_train = score_train[y_train == 0]
sig_test  = score_test[y_test == 1]
bkg_test  = score_test[y_test == 0]

ks_sig, p_sig = ks_2samp(sig_train, sig_test)
ks_bkg, p_bkg = ks_2samp(bkg_train, bkg_test)

print("=" * 55)
print("Kolmogorov–Smirnov Overtraining Test")
print("=" * 55)
print(f"  Signal:     KS stat = {ks_sig:.4f}   p-value = {p_sig:.3e}")
print(f"  Background: KS stat = {ks_bkg:.4f}   p-value = {p_bkg:.3e}")
print()
P_THRESH = 0.05
sig_flag = "⚠ OVERTRAINING" if p_sig < P_THRESH else "✓ OK"
bkg_flag = "⚠ OVERTRAINING" if p_bkg < P_THRESH else "✓ OK"
print(f"  Signal     p < {P_THRESH}? {sig_flag}")
print(f"  Background p < {P_THRESH}? {bkg_flag}")

bins = np.linspace(0, 1, 40)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_cfg = [
    (sig_train, sig_test, "royalblue", "Signal (Higgs)",       ks_sig, p_sig),
    (bkg_train, bkg_test, "tomato",    "Background (DY+Top+WW)", ks_bkg, p_bkg),
]

for ax, (tr, te, color, title, ks, pv) in zip(axes, plot_cfg):
    # Training: filled step histogram
    ax.hist(tr, bins=bins, density=True,
            histtype="stepfilled", alpha=0.40, color=color, label="Train")
    ax.hist(tr, bins=bins, density=True,
            histtype="step", lw=1.5, color=color)

    # Test: dots with Poisson error bars (standard HEP convention)
    counts, edges = np.histogram(te, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = np.diff(edges)[0]
    # Poisson error on density: sqrt(n) / (N * bin_width)
    errors = np.where(
        counts > 0,
        np.sqrt(counts / (len(te) * bin_width)),
        0
    )
    ax.errorbar(centers, counts, yerr=errors,
                fmt="o", ms=4, color=color, label="Test")

    ax.set_xlabel("RF Classifier Score  P(Higgs | event)", fontsize=11)
    ax.set_ylabel("Normalised density", fontsize=11)
    ax.set_title(
        f"{title}\nKS stat = {ks:.4f}   p-value = {pv:.3e}",
        fontsize=11
    )
    ax.legend(fontsize=10)
    ax.grid(alpha=0.25)

fig.suptitle(
    "Overtraining Check: Train (filled) vs Test (dots)\n"
    "Good model: dots follow histogram closely; p-value > 0.05",
    fontsize=12
)
fig.tight_layout()
plt.savefig("overtraining_ks_check.png", dpi=150)
plt.show()

In [ ]:
importances = rf.feature_importances_
std         = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
indices     = np.argsort(importances)[::-1]  # sort descending

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    range(len(ALL_FEATURES)),
    importances[indices],
    yerr    = std[indices],
    capsize = 4,
    color   = "steelblue",
    alpha   = 0.85,
    edgecolor="white",
)
ax.set_xticks(range(len(ALL_FEATURES)))
ax.set_xticklabels(
    [ALL_FEATURES[i] for i in indices],
    rotation=45, ha="right", fontsize=10
)
ax.set_ylabel("Gini Importance (Mean Decrease Impurity)", fontsize=11)
ax.set_title("Random Forest Feature Importance", fontsize=13)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

print("Ranked feature importances:")
for rank, idx in enumerate(indices):
    bar = "█" * int(importances[idx] * 300)
    print(f"  {rank+1:2d}. {ALL_FEATURES[idx]:25s} {importances[idx]:.4f} ± {std[idx]:.4f}  {bar}")